# LGBM Ranker: Effect of `user_aisle_pacing_ratio`
Compares the LGBM lambdarank model trained without and with `user_aisle_pacing_ratio` on the same 5% user sample and the same train/validation split.
Both models use the screening baseline settings (`feature_eda_screening.ipynb`) and early stopping, with no tuning. Feature lists come from `select_features`.

Requires BigQuery access to `instacart-ml-model.ml_model_feature_table.final_ml_features_table` and a GPU.

In [ ]:
import os
import sys

import bigframes.pandas as bpd
import lightgbm as lgb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
if project_root not in sys.path:
    sys.path.append(project_root)

from config.config import source_loc

sys.path.insert(0, source_loc)
from utilities.utility_functions import select_features, calculate_ndcg_at_k

import warnings
warnings.filterwarnings('ignore')

### 1. Load a 5% user sample

In [ ]:
bpd.options.bigquery.project = "instacart-ml-model"
bpd.options.bigquery.location = "US"

bf_full = bpd.read_gbq("instacart-ml-model.ml_model_feature_table.final_ml_features_table")

# Sample by user (never by row) so a user's orders stay together
unique_users = bf_full[['user_id']].drop_duplicates()
sample_users = unique_users.sample(frac=0.05, random_state=42)
df_sample = bf_full.merge(sample_users, on='user_id', how='inner').to_pandas()

print(f"Sample data shape: {df_sample.shape}")

### 2. Feature lists from the repo and the train/validation split
Same split as the screening notebook: 80% of users train, 20% validate.

In [ ]:
df_before, features_before, sort_list = select_features(df_sample)
df_after, features_after, _ = select_features(df_sample, include_aisle_pacing=True)

assert features_after == features_before + ['user_aisle_pacing_ratio']
assert df_sample['user_aisle_pacing_ratio'].notna().all()

train_users = pd.Series(df_sample['user_id'].unique()).sample(frac=0.8, random_state=42)
is_train = df_after['user_id'].isin(train_users)

df_train = df_after[is_train].sort_values(by=sort_list).reset_index(drop=True)
df_val = df_after[~is_train].sort_values(by=sort_list).reset_index(drop=True)

assert not set(df_train['user_id']) & set(df_val['user_id'])

# Group sizes (candidates per order) for the lambdarank model
size_array_train = df_train.groupby(sort_list, sort=False).size().to_numpy()
size_array_val = df_val.groupby(sort_list, sort=False).size().to_numpy()

df_train.shape, df_val.shape

### 3. Train both models and score ndcg@5 per order
Orders with one or no reorders are ignored, as in the screening notebook.

In [ ]:
def train_and_score(feature_list):
    """Train a ranker on feature_list with the screening baseline settings, return the model and validation ndcg@5."""
    model = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        importance_type='gain',
        device="gpu",
        n_estimators=1000,
        learning_rate=0.05
    )
    model.fit(df_train[feature_list], df_train['label_reordered'], group=size_array_train,
        eval_set=[(df_val[feature_list], df_val['label_reordered'])],
        eval_group=[size_array_val],
        callbacks=[
            lgb.early_stopping(stopping_rounds=30),
            lgb.log_evaluation(period=50)
        ]
    )
    df_scored = df_val.assign(ranking_score=model.predict(df_val[feature_list]))
    ndcg = calculate_ndcg_at_k(df_scored, 'ranking_score', 'label_reordered', sort_list, k=5)
    return model, ndcg

ranker_before, ndcg_before = train_and_score(features_before)
ranker_after, ndcg_after = train_and_score(features_after)

print(f"ndcg@5 without user_aisle_pacing_ratio: {ndcg_before:.5f}")
print(f"ndcg@5 with user_aisle_pacing_ratio:    {ndcg_after:.5f}")
print(f"Difference (with - without):            {ndcg_after - ndcg_before:+.5f}")

### 4. Feature importance of the model with `user_aisle_pacing_ratio`

In [ ]:
feature_importance = pd.DataFrame({
    'feature': features_after,
    'importance': ranker_after.feature_importances_
}).sort_values(by='importance', ascending=False).reset_index(drop=True)

feature_importance['cumulative_sum'] = feature_importance['importance'].cumsum()
feature_importance['cumulative_percentage'] = 100 * feature_importance['cumulative_sum'] / feature_importance['importance'].sum()
display(feature_importance)

new_feature_row = feature_importance[feature_importance['feature'] == 'user_aisle_pacing_ratio'].iloc[0]
new_feature_share = 100 * new_feature_row['importance'] / feature_importance['importance'].sum()
print(f"user_aisle_pacing_ratio ranks {new_feature_row.name + 1} of {len(feature_importance)} "
      f"with {new_feature_share:.2f}% of total gain")

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 8))
ax = sns.barplot(
    data=feature_importance,
    x='importance',
    y='feature',
    color="#0056b3",
    edgecolor="black",
    linewidth=0.5
)
plt.title("Feature Importance (Gain), LGBM Ranker with user_aisle_pacing_ratio", fontsize=16, fontweight="bold", pad=15)
plt.xlabel("Importance", fontsize=12, fontweight="medium", labelpad=10)
plt.ylabel("Feature Name", fontsize=12, fontweight="medium", labelpad=10)
sns.despine(top=True, right=True)
plt.show()